In [29]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
gemini_api_key = os.environ.get("GEMINI_API_KEY")
model_name = os.environ.get("GEMINI_MODEL")

In [30]:
from langchain_google_genai import GoogleGenerativeAI
model = GoogleGenerativeAI(model=model_name, api_key=gemini_api_key)

## RunnablePassthrough
It does not do anything to the input data.
Let's see it in a very simple example: a chain with just RunnablePassthrough() will output the original input without any modification.

In [31]:
from langchain_core.runnables import RunnablePassthrough

runnable = RunnablePassthrough()
runnable.invoke("Hello, world!")

'Hello, world!'

## RunnableLambda
To use a custom function inside a LCEL chain we need to wrap it up with RunnableLambda.
Let's define a very simple function to create Russian lastnames:

In [32]:
def russian_last_name(name: str) -> str:
    return f"{name}ovich"

from langchain_core.runnables import RunnablePassthrough, RunnableLambda

chain = RunnablePassthrough() | RunnableLambda(russian_last_name)
chain.invoke("Ivan")

'Ivanovich'

## RunnableParallel
We will use RunnableParallel() for running tasks in parallel.
This is probably the most important and most useful Runnable from LangChain.
In the following chain, RunnableParallel is going to run these two tasks in parallel:
operation_a will use RunnablePassthrough.
operation_b will use RunnableLambda with the russian_lastname function.

In [33]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

parallelChain = RunnableParallel(
    {
        "operation1": RunnablePassthrough(),
        "operation2": RunnableLambda(russian_last_name)
    }
)
parallelChain.invoke("Ivan")


{'operation1': 'Ivan', 'operation2': 'Ivanovich'}

In [34]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Tell me curious fact about, {soccer_player}!")
output_parser = StrOutputParser()



In [35]:
def russian_last_name_from_dictionary(person):
    return person['name'] + "ovich"



In [36]:
chains = RunnableParallel(
    {
        "operation1": RunnablePassthrough(),
        "soccer_player": RunnableLambda(russian_last_name_from_dictionary),
        "operation3": RunnablePassthrough()
    }
) | prompt | model| output_parser


In [37]:
chains.invoke({"name": "Ivan", "name1": "Petrov"})

'Because "Ivanovich" is a patronymic (meaning "son of Ivan") rather than a single historical figure, it applies to many famous Russians! \n\nHere is a curious fact about the most famous **Ivanovich** in history: **Grand Prince Ivan III Ivanovich** (better known as **Ivan the Great**).\n\n**The Curious Fact:**\nIvan the Great is the man who essentially laid the foundations for the modern Russian state, but he accidentally created the title **"Czar"** (derived from "Caesar") through a massive diplomatic flex and a very clever marriage. \n\nAfter the fall of Constantinople in 1453, the Byzantine Empire was no more. To legitimize Moscow as the "Third Rome" and the protector of the Orthodox Christian world, Ivan married **Sophia Paleologue**, the niece of the last Byzantine Emperor. \n\nTo impress the snooty European monarchs who still viewed Russia as a backward, provincial backwater, Ivan started using the title *Czar* and adopted the **two-headed Byzantine eagle** as Russia’s coat of arm

### Let's see a more advanced use of RunnableParallel

In [40]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.output_parsers import StrOutputParser


embeddingsModel = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

chunk_of_text = "Virat Kohli is an Indian cricketer. He is widely regarded as one of the best batsmen in the world. He has numerous records and accolades to his name. He is known for his aggressive batting style and consistency in international cricket. His dedication and fitness regime have set new standards in the sport. His wife popular actress Anushka Sharma."

vector_store = Chroma.from_texts([chunk_of_text], embeddingsModel, persist_directory="chroma_db")

retriver = vector_store.as_retriever()
output_parser = StrOutputParser()






In [41]:
template = "Answer the following question based on the given context: {context}\nQuestion: {question}"

prompt = ChatPromptTemplate.from_template(template)

retrival_chain = (
    RunnableParallel({"context": retriver, "question": RunnablePassthrough()})
    | prompt
    | model
    | output_parser
                  
                  )

In [42]:
retrival_chain.invoke("Who is Virat Kohli?")

'Based on the provided context, Virat Kohli is an Indian cricketer who is widely regarded as one of the best batsmen in the world. He is known for his aggressive batting style, consistency in international cricket, numerous records and accolades, and his dedication and fitness regime. Additionally, his wife is the popular actress Anushka Sharma.'

In [44]:
from operator import itemgetter

template = "Answer the following question based on the given context: {context}\nQuestion: {question} \n Answer in the following language: {language}"

prompt = ChatPromptTemplate.from_template(template)

chainsOfParallel = ({
    "context": itemgetter("question") | retriver,
    "question": itemgetter("question"),
    "language": itemgetter("language")
}                 
| prompt
| model
| output_parser
)

In [50]:
chainsOfParallel.invoke({"question": "Who is Virat Kohli?", "language": "Pirate English"})

"Ahoy, matey! Gather 'round and listen well! \n\nVirat Kohli be a mighty fine Indian cricketer, widely feared and respected as one of the very best batsmen to ever walk the deck of international cricket! He’s got more treasures (records and accolades) to his name than ye can shake a cutlass at, known far and wide for his fiercely aggressive batting style and steady hand. Plus, his relentless training and ship-shape fitness have set a whole new standard across the high seas of sport! Oh, and he shares his quarters with his bonnie wife, the popular actress Anushka Sharma! Arrr!"